Connect to dataset on google drive if you want to run notebook on google colab

In [1]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In next cells, we extract frames of each video based on face emotional changes in the video. Additionally, audio extraction is performed.

You can run this notebook on train set and test sets to apply the same preprocessing on both of them.

In [3]:
import cv2
from fer import FER
import os

def get_max_emotions_and_save_frames(vid, videos_folder, output_folder):
    # Load an FER detector
    detector = FER()

    # Open the video file
    cap = cv2.VideoCapture(os.path.join(videos_folder, vid))

    # Read the first frame
    ret, frame = cap.read()

    # Initialize variables to track the last max emotion
    last_max_emotion = None

    # Read frames and detect emotions
    frame_number = 0
    while True:
        ret, frame = cap.read()

        # Break the loop if no more frames
        if not ret:
            break

        # Convert the frame to RGB (fer library uses RGB)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        try:

          # Detect emotions in the frame
          emotions_list = detector.detect_emotions(rgb_frame)

          # Check if any faces are detected
          if emotions_list:
              # Get the emotion dictionary for the face with the maximum score
              max_face_emotion = max(emotions_list, key=lambda x: max(x['emotions'].values()))

              # Get the emotion label with the maximum score
              max_emotion_label = max(max_face_emotion['emotions'], key=max_face_emotion['emotions'].get)

              # Get the maximum score for the detected emotion
              max_score = max_face_emotion['emotions'][max_emotion_label]

              if frame_number == 0:
                  # Save the selected face in size 128x128
                  x, y, w, h = max_face_emotion['box']
                  x1, y1, x2, y2 = x - 10, y + 10, x - 10 + w + 20, y + 10 + h
                  face = frame[y1:y2, x1:x2]
                  face_resized = cv2.resize(face, (128, 128))
                  cv2.imwrite(os.path.join(output_folder, vid.split('.')[0] + '_' + str(frame_number) + '.png'), face_resized)
                  # Update the last max emotion
                  last_max_emotion = max_emotion_label
                  frame_number += 1
              else:
                  # Check if the new max emotion is different from the previous emotion
                  if max_emotion_label != last_max_emotion:
                      # Save the selected face in size 128x128
                      x, y, w, h = max_face_emotion['box']
                      x1, y1, x2, y2 = x - 10, y + 10, x - 10 + w + 20, y + 10 + h
                      face = frame[y1:y2, x1:x2]
                      # Check if the face image is not empty
                      if face is not None and face.size > 0:
                          face_resized = cv2.resize(face, (128, 128))
                          # Save the resized face image
                          cv2.imwrite(os.path.join(output_folder, vid.split('.')[0] + '_' + str(frame_number) + '.png'), face_resized)
                          print(f"Saved frame with emotion: {max_emotion_label}")
                      # Update the last max emotion
                      last_max_emotion = max_emotion_label
                      # Increment frame number
                      frame_number += 1
        except Exception as e:
          # Handle the exception (e.g., print an error message)
          print(f"Error processing frame: {str(e)}")

    # Release the video capture object
    cap.release()

    print(f"Generated {frame_number} frames for {vid}")



In [ ]:
from moviepy.video.io.VideoFileClip import VideoFileClip
import os

def extract_and_save_audio(vid, videos_folder, output_audio_folder):
    """Extract audio from video and save as FLAC."""
    input_video_path = os.path.join(videos_folder, vid)
    video_clip = VideoFileClip(input_video_path)
    audio_clip = video_clip.audio
    audio_output_path = os.path.join(output_audio_folder, os.path.splitext(os.path.basename(input_video_path))[0] + ".flac")

    # Save audio in FLAC format
    audio_clip.write_audiofile(audio_output_path, codec='flac')

    # Close the clips to release resources
    video_clip.close()
    audio_clip.close()

In [4]:
def create_folder(folder_path):
    """Create a folder if it doesn't exist."""
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

In [ ]:
import os
import cv2
import shutil
import gc
import re
import json
from pylab import *
from PIL import Image, ImageChops, ImageEnhance
import scipy
import scipy.misc
from scipy import ndimage
import skimage.io
import skimage.filters
import sys

#loop throght videos in each folder of the four folders and extract frames
videos_folder = '/content/drive/MyDrive/RARV'
output_frames_folder = '/content/drive/MyDrive/RARV_frames'
moved_videos_folder = '/content/drive/MyDrive/RARV_moved'
output_audio_folder = '/content/drive/MyDrive/RARV_audio'

# Create output folders if they don't exist
create_folder(output_frames_folder)
create_folder(moved_videos_folder)
create_folder(output_audio_folder)

list_of_train_data = [f for f in os.listdir(videos_folder) if f.endswith('.mp4')]

for vid in list_of_train_data:
  get_max_emotions_and_save_frames(vid, videos_folder, output_frames_folder)
  extract_and_save_audio(vid, videos_folder, output_audio_folder)

# Close all OpenCV windows
cv2.destroyAllWindows()

# Move the video file to the "moved_videos" folder
video_path = os.path.join(videos_folder, vid)
moved_video_path = os.path.join(moved_videos_folder, vid)
try:
    shutil.move(video_path, moved_video_path)
    print(f"Moved video: {vid} to {moved_video_path}")
except Exception as e:
    print(f"Error moving video {vid}: {str(e)}")


In [ ]:
import os
import cv2
import shutil
import gc
import re
import json
from pylab import *
from PIL import Image, ImageChops, ImageEnhance
import scipy
import scipy.misc
from scipy import ndimage
import skimage.io
import skimage.filters
import sys


#loop throght videos in each folder of the four folders and extract frames
videos_folder = '/content/drive/MyDrive/FARV'
output_frames_folder = '/content/drive/MyDrive/FARV_frames'
moved_videos_folder = '/content/drive/MyDrive/FARV_moved'
output_audio_folder = '/content/drive/MyDrive/FARV_audio'

# Create output folders if they don't exist
create_folder(output_frames_folder)
create_folder(moved_videos_folder)
create_folder(output_audio_folder)

list_of_train_data = [f for f in os.listdir(videos_folder) if f.endswith('.mp4')]

for vid in list_of_train_data:
  get_max_emotions_and_save_frames(vid, videos_folder, output_frames_folder)
  extract_and_save_audio(vid, videos_folder, output_audio_folder)

# Close all OpenCV windows
cv2.destroyAllWindows()

# Move the video file to the "moved_videos" folder
video_path = os.path.join(videos_folder, vid)
moved_video_path = os.path.join(moved_videos_folder, vid)
try:
    shutil.move(video_path, moved_video_path)
    print(f"Moved video: {vid} to {moved_video_path}")
except Exception as e:
    print(f"Error moving video {vid}: {str(e)}")


In [ ]:
import os
import cv2
import shutil
import gc
import re
import json
from pylab import *
from PIL import Image, ImageChops, ImageEnhance
import scipy
import scipy.misc
from scipy import ndimage
import skimage.io
import skimage.filters
import sys

#loop throght videos in each folder of the four folders and extract frames
videos_folder = '/content/drive/MyDrive/RAFV'
output_frames_folder = '/content/drive/MyDrive/RAFV_frames'
moved_videos_folder = '/content/drive/MyDrive/RAFV_moved'
output_audio_folder = '/content/drive/MyDrive/RAFV_audio'

# Create output folders if they don't exist
create_folder(output_frames_folder)
create_folder(moved_videos_folder)
create_folder(output_audio_folder)

list_of_train_data = [f for f in os.listdir(videos_folder) if f.endswith('.mp4')]

for vid in list_of_train_data:
  get_max_emotions_and_save_frames(vid, videos_folder, output_frames_folder)
  extract_and_save_audio(vid, videos_folder, output_audio_folder)

# Close all OpenCV windows
cv2.destroyAllWindows()

# Move the video file to the "moved_videos" folder
video_path = os.path.join(videos_folder, vid)
moved_video_path = os.path.join(moved_videos_folder, vid)
try:
    shutil.move(video_path, moved_video_path)
    print(f"Moved video: {vid} to {moved_video_path}")
except Exception as e:
    print(f"Error moving video {vid}: {str(e)}")


In [ ]:
import os
import cv2
import shutil
import gc
import re
import json
from pylab import *
from PIL import Image, ImageChops, ImageEnhance
import scipy
import scipy.misc
from scipy import ndimage
import skimage.io
import skimage.filters
import sys

#loop throght videos in each folder of the four folders and extract frames
videos_folder = '/content/drive/MyDrive/FAFV'
output_frames_folder = '/content/drive/MyDrive/FAFV_frames'
moved_videos_folder = '/content/drive/MyDrive/FAFV_moved'
output_audio_folder = '/content/drive/MyDrive/FAFV_audio'

# Create output folders if they don't exist
create_folder(output_frames_folder)
create_folder(moved_videos_folder)
create_folder(output_audio_folder)

list_of_train_data = [f for f in os.listdir(videos_folder) if f.endswith('.mp4')]

for vid in list_of_train_data:
  get_max_emotions_and_save_frames(vid, videos_folder, output_frames_folder)
  extract_and_save_audio(vid, videos_folder, output_audio_folder)

# Close all OpenCV windows
cv2.destroyAllWindows()

# Move the video file to the "moved_videos" folder
video_path = os.path.join(videos_folder, vid)
moved_video_path = os.path.join(moved_videos_folder, vid)
try:
    shutil.move(video_path, moved_video_path)
    print(f"Moved video: {vid} to {moved_video_path}")
except Exception as e:
    print(f"Error moving video {vid}: {str(e)}")
